In [2]:
from model_builder import ModelBuilder
from model_builder_keras import KerasModelBuilder

from preprocessing import Preprocessor
from plotting_other import Plotter
from plotting import plot_dataset
#from shapley import ProcessAttributor
from shapley_improved import ProcessAttributorSHAP
from shapley_improved_other import ProcessAttributorSHAPMLP

from universal_filtering import CustomSpearmanFilter
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold, SelectFromModel
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
#from sklearn.linear_model import Ridge
#from sklearn.linear_model import Lasso
from sklearn.base import BaseEstimator, RegressorMixin
import numpy as np

# Basic Deep Learning with Sklearn MLP
from sklearn.neural_network import MLPRegressor
from sklearn.inspection import permutation_importance

# Deep Learning with Keras Tensorflow
#import keras
from keras import layers, optimizers, callbacks, Sequential
#https://ipython.org/ipython-doc/3/config/extensions/autoreload.html
%load_ext autoreload
%autoreload 2

In [3]:
train_sarek = [
    pd.read_parquet("../../../../ProcessEnergyAccounting/runs/nfcore-20260701T215234Z/datasets/sarek_1_0207.parquet"),
    pd.read_parquet("../../../../ProcessEnergyAccounting/runs/nfcore-20260702T193504Z/datasets/sarek_2_0207.parquet")
]
test_sarek = pd.read_parquet("../../../../ProcessEnergyAccounting/runs/nfcore-20260708T212252Z/datasets/sarek3_0907.parquet")


In [4]:
training_data = pd.concat(train_sarek, ignore_index=True)
training_data = training_data
test_data = test_sarek
PNG_NAME = "mlp_pred_sarek"

features = [
    "delta_cpu_ns",
    "delta_io_bytes",
    "delta_net_send_bytes",
    "context_switches",
    "syscall_count",
    "delta_rss_memory",
    "delta_cpu_time_psutil",
    "delta_cpu_time_proc",
    "syscall_class_file",
    "syscall_class_network",
    "syscall_class_memory",
    "syscall_class_process",
    "syscall_class_other",
    "syscall_class_sched",
    "syscall_class_signal",
    "syscall_class_time",
    "delta_cycles",
    "delta_cache_misses",
    "delta_instructions",
    "delta_branch_instructions",
]


In [5]:
preprocessor_train = Preprocessor(training_data, features)
X_train_FULL, y_train, t_train, _ = preprocessor_train.preprocess_no_split()

Dropped 0 timestamps.


In [6]:
#plot_dataset(t_train, y_train, "multi_training")


In [7]:
#build_model = ExplainableBoostingRegressor( interactions=2, max_rounds=2000, n_jobs=-1, random_state=42)
model = RandomForestRegressor(n_estimators=100,  n_jobs=-1, random_state=42)
#build_model = MLPRegressor(activation="relu", solver="adam", random_state=42)



In [8]:
#These thresholds could be fine tuned
automatic_feature_selection = Pipeline(steps=[
    ('variance', VarianceThreshold(threshold=0.01)), #explain this

    ('decorrelate', CustomSpearmanFilter(threshold=0.80)),
    ('scaler', StandardScaler()),
    ('select_features', SelectFromModel(model, threshold='0.5*median'))
])

automatic_feature_selection.set_output(transform="pandas")
automatic_feature_selection.fit_transform(X_train_FULL, y_train)
good_features = automatic_feature_selection.get_feature_names_out().tolist()
X_train = X_train_FULL[good_features]
print("Selected columns:")
print(good_features)


#plot_dataset(t_train, y_train, "multi_training")

Selected columns:
['delta_cpu_ns', 'delta_io_bytes', 'delta_net_send_bytes', 'context_switches', 'syscall_count', 'delta_rss_memory', 'delta_cpu_time_proc', 'syscall_class_file', 'syscall_class_network', 'syscall_class_memory', 'syscall_class_process', 'syscall_class_other']


In [9]:
preprocessor_test = Preprocessor(test_data, good_features)
X_test, y_test, t_test , X_test_unaggregated = preprocessor_test.preprocess_no_split()

#plot_dataset(t_test, y_test, "multi_testing")

Dropped 0 timestamps.


In [10]:
plot_dataset(t_test, y_test, "multi_testing")

### Sklearn MLP

### Deep Learning with Keras

Build the 1D-CNN in Keras (Tutorials)

- https://keras.io/guides/sequential_model/
- https://keras.io/examples/timeseries/timeseries_classification_from_scratch/
- https://www.tensorflow.org/tutorials/structured_data/time_series#recurrent_neural_network
- https://keras.io/examples/timeseries/timeseries_classification_from_scratch/

Different: Add windowing
- https://www.tensorflow.org/tutorials/structured_data/time_series#2_split
- https://numpy.org/doc/stable/reference/generated/numpy.lib.stride_tricks.sliding_window_view.html 
- Pay attention to window_size in layers.Input(shape=(num_features, window_size))
- The data shape is (batch_size, window, features).

In [ ]:
num_features = len(good_features)
window_size = 20 # Initial 1

In [20]:
# Models used
# Convolutional Neural Network (1D)
cnn_model = Sequential([

    layers.Input(shape=(num_features, window_size)), # (num_features, sequence_length) #Only current value
    layers.Conv1D(32, kernel_size=num_features, padding='same', activation="relu"),
    layers.BatchNormalization(),

    layers.Conv1D(32, kernel_size=num_features, padding='same', activation="relu"),
    layers.BatchNormalization(),

    #layers.Conv1D(32, kernel_size=num_features, padding='same', activation="relu"),
    #layers.BatchNormalization(),
    
    layers.Flatten(),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
    
])

# LSTM Model
lstm_model = Sequential([
    layers.Input(shape=(num_features, window_size)), # (num_features, sequence_length) #Only current value
    #layers.BatchNormalization(),
    layers.LSTM(64, return_sequences=True),
    layers.LSTM(64, return_sequences=True),
    layers.Flatten(),
    #layers.GlobalAveragePooling1D(),
    #layers.Dropout(0.2),
    #layers.Dense(32, activation='relu'),
    layers.Dense(1)
    
])

In [21]:
lstm_model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_4 (LSTM)                   │ (None, 12, 64)         │        21,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 12, 64)         │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_5 (Flatten)             │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │           769 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 55,553 (217.00 KB)

 Trainable params: 55,553 (217.00 KB)

 Non-trainable params: 0 (0.00 B)

In [28]:
builder_lstm = KerasModelBuilder(X_train, X_test, y_train, y_test, lstm_model, StandardScaler(), 
                                 window_size=window_size, train_epochs=1)
y_pred_lstm, learned_idle_power = builder_lstm.run_and_save_model()

209/209 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - loss: 58.7089 - mae: 4.9839 - val_loss: 31.5758 - val_mae: 2.8500
260/260 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
  R² Score:  0.9196
  MAE:       5.40 Ws (2.68% of mean)
----------------------------------


ValueError: window shape cannot be larger than input array shape

In [ ]:
#The data shape is (batch_size, window, features). 
builder_lstm.idle_power()

In [43]:
array = [[-7.59477105e-01, -2.38305630e-01]]
array_t = np.tile(array, (10,1))
array_t = np.expand_dims(array_t, axis=0)
array_t.shape


(1, 10, 2)

In [ ]:
#plotter = Plotter(y_pred=y_pred_lstm,y_test=y_test, t_test= t_test,alg_name="lstm",window_start=150, window_end=200)
plotter = Plotter(y_pred=y_pred_lstm,y_test=y_test, t_test= t_test,alg_name="lstm")
#plotter.plot_only("lstm_")
plotter.plot_and_save("ltsm_windowing_")

In [69]:
builder_cnn = KerasModelBuilder(X_train, X_test, y_train, y_test, cnn_model, StandardScaler(), window_size=window_size, 
                                train_epochs=20)
y_pred_cnn, learned_idle_power = builder_cnn.run_and_save_model()

Epoch 1/20
209/209 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 14052.0273 - mae: 87.7728 - val_loss: 1773.4613 - val_mae: 39.7536
Epoch 2/20
209/209 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 129.1358 - mae: 8.2267 - val_loss: 85.0862 - val_mae: 5.0218
Epoch 3/20
209/209 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 101.4807 - mae: 7.3225 - val_loss: 97.8158 - val_mae: 6.8453
Epoch 4/20
209/209 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 89.3396 - mae: 6.7501 - val_loss: 72.8178 - val_mae: 5.1984
Epoch 5/20
209/209 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 77.2737 - mae: 6.2355 - val_loss: 56.9133 - val_mae: 4.2397
Epoch 6/20
209/209 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 75.7953 - mae: 6.1527 - val_loss: 49.8172 - val_mae: 3.7623
Epoch 7/20
209/209 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 68.5168 - mae: 5.8224 - val_loss: 46.7333 - val_mae: 3.8045
Epoch 8/20
209/209 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 67.6139 - mae: 5.7881 - val_loss: 60.9996 - val_mae: 5.3552
Epoch 9/20
209/209 ━━━━━━━━━━━━

ValueError: Exception encountered when calling Sequential.call().

[1mCannot take the length of shape with unknown rank.[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=<unknown>, dtype=float32)
  • training=False
  • mask=None
  • kwargs=<class 'inspect._empty'>

In [ ]:
plotter = Plotter(y_pred=y_pred_cnn,y_test=y_test, t_test= t_test,alg_name="cnn_1d")
#plotter.plot_only("cnn_1d")
plotter.plot_and_save("cnn_1d_")

In [50]:
builder_rf = ModelBuilder(X_train, X_test, y_train, y_test, model, StandardScaler())
y_pred_rf, learned_idle_power = builder_rf.run_and_save_model()


  R² Score:  0.9292
  MAE:       5.17 Ws (2.57% of mean)
----------------------------------
The model's learned baseline idle interval energy is: 141.68 Ws
----------------------------------
/n


In [62]:
builder_rf.idle_power()

(8338, 12)
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]
[[-7.59477105e-01 -2.38305630e-01 -4.07955034e+00 -5.23644059e-01
  -1.08708467e+00 -2.65926613e-03 -1.00288252e+00 -7.33254323e-01
  -4.49093552e-01 -2.22468560e-01 -4.01756529e-01 -8.78905289e-01]]
(1, 12)
[141.67764629]
The model's learned baseline idle interval energy is: 141.68 Ws
----------------------------------
/n
